In [38]:
import polars as pl
import numpy as np
import sys
sys.path.append("/kaggle/input/models/fedorfesenko/metrcs/other/default/1")

from metrics import evaluate_candidates

In [39]:
TOP_K = 500


In [40]:
df_interactions_train = pl.read_parquet("/kaggle/input/datasets/fedorfesenko/train-interactions/df_interactions_train.parquet")
df_interactions_val = pl.read_parquet("/kaggle/input/datasets/fedorfesenko/train-interactions/df_interactions_val.parquet")

In [41]:
pop = (
    df_interactions_train
    .filter(pl.col("action_type") == "AT_CartUpdate")
    .group_by(pl.col("product_id"))
    .agg(pl.len().alias("n_carts"))
    .sort(by="n_carts", descending=True)
)
pop.head(20)

product_id,n_carts
u64,u32
17534690445912077162,1134
9080005013554695731,922
3246920192090166857,827
14589993416551700732,773
14450414801503444571,691
…,…
12811420339782479517,510
1050376813224499185,506
11465417520669799786,472


In [42]:
top_items = pop.head(TOP_K)["product_id"].to_list()

In [43]:
true_items_interactions_df = (
    df_interactions_val.filter(pl.col("action_type") == "AT_CartUpdate")
    .group_by("user_id")
    .agg(pl.col("product_id").alias("true_items"))
)
true_items_interactions = {
    row["user_id"]: row["true_items"]
    for row in true_items_interactions_df.iter_rows(named=True)
}

In [44]:
predicted_interactions = {
    user_id: top_items
    for user_id in true_items_interactions.keys()
}

In [45]:
result = evaluate_candidates(
    predicted_interactions,
    true_items_interactions,
    TOP_K,
    total_items=df_interactions_train["product_id"].n_unique()
)

In [46]:
result

{'recall@500': 0.23250329899113928,
 'hit_rate@500': 0.7824074074074074,
 'mrr@500': 0.08734376030074975,
 'coverage': 0.01896165952444158,
 'n_requests': 432,
 'avg_candidates': 500.0}